# 13 – Service Layer (Mock vs Real, Factory Pattern)

The service factory provides a clean mock/real switching point via `ENABLE_MOCK` env var.  
This notebook tests all four mock services directly and shows how to swap them out.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
os.environ['ENABLE_MOCK'] = 'true'

## 1. Factory — service selection

In [ ]:
from services.factory import get_data_service, get_ticket_service, get_metadata_service, get_vector_service

data_svc    = get_data_service()
ticket_svc  = get_ticket_service()
meta_svc    = get_metadata_service()
vector_svc  = get_vector_service()

print('data_service    :', type(data_svc).__name__)
print('ticket_service  :', type(ticket_svc).__name__)
print('metadata_service:', type(meta_svc).__name__)
print('vector_service  :', type(vector_svc).__name__)

## 2. MockDatabricksService — SQL query simulation

In [ ]:
from services.databricks.mock import MockDatabricksService

svc = MockDatabricksService()

queries = [
    "SELECT * FROM analytics.retention_metrics WHERE period = 'last_30_days' LIMIT 1",
    "SELECT * FROM analytics.bookings_fact WHERE period = 'last_30_days' LIMIT 1",
    "SELECT * FROM analytics.cac_metrics WHERE period = 'last_30_days' LIMIT 1",
    "SELECT * FROM analytics.customer_ltv WHERE period = 'last_30_days' LIMIT 1",
]

for sql in queries:
    rows = svc.query(sql)
    table = sql.split('FROM')[1].split('WHERE')[0].strip()
    print(f'Table: {table}')
    if rows:
        print(f'  Row: {rows[0]}')
    print()

## 3. MockDatabricksService — low GRR mode

In [ ]:
svc_low = MockDatabricksService(low_grr=True)
rows = svc_low.query("SELECT * FROM analytics.retention_metrics")
print('Low GRR mode — retention:', rows[0])

## 4. MockJiraService — issues and creation

In [ ]:
from services.jira.mock import MockJiraService

jira = MockJiraService()

# Search
issues = jira.search_issues('project = DGC', max_results=3)
print(f'Found {len(issues)} issues:')
for i in issues:
    f = i['fields']
    print(f"  {i['key']}  [{f['priority']['name']}]  {f['summary']}")

print()

# Create
new = jira.create_issue(
    summary='Test: retention anomaly',
    description='GRR fell to 72% in EMEA',
    issue_type='Bug',
    priority='High',
    labels=['retention', 'auto-generated'],
)
print(f"Created ticket: {new['key']} — {new['fields']['summary']}")

## 5. MockCollibraService — asset search and DQ

In [ ]:
from services.collibra.mock import MockCollibraService

collibra = MockCollibraService()

for product in ['retention', 'bookings', 'cac', 'ltv']:
    assets = collibra.search_assets(product)
    if assets:
        a = assets[0]
        dq = collibra.get_data_quality(a['id'])
        print(f"{product:<12}  asset={a['name']:<20}  owner={a['owner']:<30}  DQ={dq['score']}%")

## 6. NullVectorService — similarity search

In [ ]:
from services.pgvector.mock import NullVectorService

vs = NullVectorService()
results = vs.similarity_search('retention churn GRR policy', k=3)

print(f'Top {len(results)} results:')
for doc, score in results:
    print(f'  score={score}  topic={doc.metadata["topic"]}')
    print(f'  content: {doc.page_content[:80]}...')
    print()